## Load Data and Libraries

In [1]:
# From Bob: I uncommented this line or else I cannot query the data in block 3.
from project import icu_query, vitals_query, labs_query

In [2]:
from project import bucket_ecg_report_0, simplify_race, simplify_careunit
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
df_icu = icu_query()
df_vitals = vitals_query()
df_labs = labs_query()
original_query = df_icu.merge(df_vitals, on='stay_id', how='left').merge(df_labs, on='stay_id', how='left')
df = original_query
display(df.shape)

/Users/jack/ucsf_courses_resources/winter 2026/datasci_223/final/venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


(35474, 45)

In [4]:
# need to categorize report_0
# getting frequency of report_0 codes
original_ecg_report0_frequency = df['report_0'].value_counts()
original_ecg_report0_frequency.to_csv('sanity_outputs/original_ecg_report0_frequency.csv')

## ECG Report Cleaning

In [5]:
# sinus rhythm, sinus tachycardia, a.fib
df['report_0'] = df['report_0'].str.lower().str.removesuffix(".").str.replace('"', '')
df = df[~df['report_0'].str.contains('warning')]
initial_drop = df['report_0'].value_counts()
initial_drop.to_csv('sanity_outputs/initial_drop.csv')
# handles: Sinus rhythm, Sinus tachycardia, Atrial fibrillation, Sinus bradycardia that had '.' suffix
# changes all to lower, drops records with 'warning' because those signals could harm our model
df = df[df['report_0'].notna()] # handles any NA values

In [6]:
# check if any NAs, there were none
print(original_query.shape)
print(df.shape)

(35474, 45)
(34830, 45)


In [7]:
# check mortality captured by these first four catgories ()
df_first_cats = df[df['report_0'].isin(['sinus rhythm', 'sinus tachycardia', 'atrial fibrillation', 'sinus bradycardia'])]
print(f"")
print(f"Mortalities captured by the first four report_0 buckets: {sum(df_first_cats['hospital_expire_flag'])} ({sum(df_first_cats['hospital_expire_flag']) / sum(df['hospital_expire_flag']):.2%})")
print(f"Total mortalities: {sum(df['hospital_expire_flag'])}")


Mortalities captured by the first four report_0 buckets: 2061 (49.13%)
Total mortalities: 4195


In [8]:
df['ecg_bucket'] = df['report_0'].apply(bucket_ecg_report_0)
# check mortality captured after bucket_ecg_report_0 function
df_cats = df[df['ecg_bucket'].isin(['normal_sinus', 'sinus_tachy', 'sinus_brady', 'afib', 'afib_rvr', 'pvc', 'pac', 'paced', 'stemi_alert', 'atrial_ectopic', 'supraventricular', 'idioventricular', 'accelerated_junctional'])]
print(f"Mortalities captured by the current buckets: {sum(df_cats['hospital_expire_flag'])} ({sum(df_cats['hospital_expire_flag']) / sum(df['hospital_expire_flag']):.2%})")
print(f"Total mortalities: {sum(df['hospital_expire_flag'])}")
post_bucket_function = df['ecg_bucket'].value_counts()
post_bucket_function.to_csv('sanity_outputs/post_bucket_function.csv')
# capturing 98.7% of mortalities in all categories != 'other'

Mortalities captured by the current buckets: 4141 (98.71%)
Total mortalities: 4195


In [9]:
named_buckets = ['normal_sinus', 'sinus_tachy', 'sinus_brady', 'afib', 'afib_rvr', 'pvc', 'pac', 'paced', 'stemi_alert', 'atrial_ectopic', 'supraventricular', 'idioventricular', 'accelerated_junctional']
df_other = df[~df['ecg_bucket'].isin(named_buckets)]
df_other.groupby('ecg_bucket')['hospital_expire_flag'].agg(['sum', 'count', 'mean']).sort_values('sum', ascending=False)

,sum,count,mean
ecg_bucket,,,
undetermined rhythm,21,143,0.146853
a-v dissociation,6,69,0.086957
wide qrs tachycardia,4,17,0.235294
"age not entered, assumed to be 50 years old for purpose of ecg interpretation",4,42,0.095238
probable junctional rhythm,4,58,0.068966
--- pediatric criteria used ---,2,31,0.064516
a-v dissociation with unclassified aberrant complexes,2,6,0.333333
probable ventricular tachycardia,2,12,0.166667
possible junctional rhythm,2,16,0.125


In [10]:
df['ecg_bucket'] = df['ecg_bucket'].where(df['ecg_bucket'].isin(named_buckets), 'other')
df['ecg_bucket'].value_counts()

ecg_bucket
normal_sinus              15670
sinus_tachy                4256
afib                       2579
sinus_brady                2508
afib_rvr                   2214
pvc                        2164
pac                        1494
paced                      1129
stemi_alert                 795
atrial_ectopic              566
other                       471
supraventricular            373
accelerated_junctional      327
idioventricular             284
Name: count, dtype: int64

In [11]:
df.to_csv('sanity_outputs/report0_cleaned.csv')
# report_0 cleaned values in new column: ecg_bucket

## Gender Encoding

In [12]:
# set gender to 0 (Male), 1 (Female)
df['gender'] = df['gender'].replace(['M', 'F'], [0,1])

/var/folders/6h/ch1ltdnj3v33nkw2k13k_h8w0000gn/T/ipykernel_30064/1839804122.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['gender'] = df['gender'].replace(['M', 'F'], [0,1])


## ECG Measurement Cleaning

In [13]:
# cleaning ecg measurements: explore columns
ecg_intervals = ['rr_interval','p_onset', 'p_end', 'qrs_onset', 'qrs_end', 't_end']
ecg_axes = ['p_axis', 'qrs_axis', 't_axis']

display(df[ecg_intervals].describe())
display(df[ecg_axes].describe())

,rr_interval,p_onset,p_end,qrs_onset,qrs_end,t_end
count,34830.0,34830.0,34830.0,34830.0,34830.0,34830.0
mean,758.970112,6748.931467,8278.493913,211.397962,312.926357,604.997387
std,306.037663,12472.402043,13292.490344,176.20035,178.651299,188.8187
min,285.0,7.0,5.0,24.0,0.0,285.0
25%,606.0,40.0,138.0,178.0,270.0,542.0
50%,731.0,40.0,154.0,200.0,294.0,592.0
75%,869.0,333.0,29999.0,214.0,328.0,646.0
max,29999.0,29999.0,29999.0,29999.0,29999.0,29999.0


,p_axis,qrs_axis,t_axis
count,34830.0,34830.0,34830.0
mean,6870.631869,92.460781,141.55234
std,12576.320922,1523.214416,1781.883849
min,-21846.0,-180.0,-180.0
25%,42.0,-17.0,6.0
50%,61.0,15.0,45.0
75%,86.0,50.0,79.0
max,32767.0,29999.0,32767.0


In [14]:
# drop p_onset, p_end, p_axis as predictor because they have so many values at 29999 ms (physiologically impossible) imputing would be dangerous
df = df.drop(columns=['p_onset', 'p_end', 'p_axis'])

# imputing impossible values with median of associated ecg_bucket value
for col in ['rr_interval', 'qrs_onset', 'qrs_end', 't_end']:
    df.loc[df[col]>4000, col] = pd.NA
for col in ['qrs_axis', 't_axis']:
    df.loc[~df[col].between(-180,180), col] = pd.NA

ecg_measurements = ['rr_interval', 'qrs_onset', 'qrs_end', 't_end', 'qrs_axis', 't_axis']
df[ecg_measurements]=df[ecg_measurements].astype(float)
df[ecg_measurements] = df.groupby('ecg_bucket')[ecg_measurements].transform(lambda x: x.fillna(x.median()))

## Categorical Feature Cleaning

In [15]:
# handle missing values: language, marital status
# handle spaces and case: language, marital status, race, admission type, admission location, first care unit, last care unit
df['language'] = df['language'].fillna('unknown')
df['language'] = df['language'].str.lower().str.replace(' ', '_')

df['marital_status'] = df['marital_status'].fillna('unknown')
df['marital_status'] = df['marital_status'].str.lower()

df['race'] = df['race'].apply(simplify_race) # function handles

df['admission_type'] = df['admission_type'].str.lower().str.replace(' ','_').str.removesuffix('.')

df['admission_location'] = df['admission_location'].str.lower().str.replace('-', '_').str.replace(' ', '_').str.replace('/', '_')
df['admission_location'].value_counts()

# there are no differences in facilities between first vs last care unit in this dataset, shrinking to 1 column: care_unit
df = df.rename(columns={'first_careunit': 'care_unit'})
df['care_unit'] = df['care_unit'].apply(simplify_careunit)
df = df.drop(columns=['last_careunit'])

# change 'anchor_age' col to 'age'
df = df.rename(columns={'anchor_age': 'age'})

## Vitals & Lab Value Cleaning

In [16]:
# set boundaries for new cols
# change temp to numeric
df['temperature'] = pd.to_numeric(df['temperature'], errors='coerce')
# spO2 cut off < 70
df.loc[df['spo2'] < 70, 'spo2'] = pd.NA
# glucose cut off <40 and >600
df.loc[~df['glucose'].between(40,600), 'glucose'] = pd.NA

# impute vitals and lab values with the median of associated ecg_bucket value
# handle missing on above columns and hr, sbp, dbp, bun, creatinine, bicarbonate, lactate

biomarkers = ['temperature', 'spo2', 'glucose', 'heart_rate', 'sbp', 'dbp', 'mbp', 'resp_rate', 'lactate', 'creatinine', 'bun', 'bicarbonate']
df[biomarkers] = df.groupby('ecg_bucket')[biomarkers].transform(lambda x: x.fillna(x.median()))
df[biomarkers] = df[biomarkers].fillna(df[biomarkers].median())

## Final Dataset & Export

In [17]:
# drop columns that will not be used in modeling
keep_cols = [
    'care_unit', 'admission_type', 'admission_location', 
    'language', 'marital_status', 'race', 'hospital_expire_flag', 
    'gender', 'age', 'rr_interval', 'qrs_onset', 'qrs_end', 't_end', 'qrs_axis', 't_axis', 'ecg_bucket',
    'temperature', 'spo2', 'glucose', 'heart_rate', 'sbp', 'dbp', 'mbp', 'resp_rate', 'lactate', 'bun',
    'creatinine', 'bicarbonate'
]
df = df[keep_cols]
df.shape

(34830, 28)

In [ ]:
# Bob: Check columns, head, and export cleaned data to csv
#print(df.columns)
#display(df.head())
df.to_csv('data/cleaned_data.csv')

Index(['care_unit', 'admission_type', 'admission_location', 'language',
       'marital_status', 'race', 'hospital_expire_flag', 'gender', 'age',
       'rr_interval', 'qrs_onset', 'qrs_end', 't_end', 'qrs_axis', 't_axis',
       'ecg_bucket', 'temperature', 'spo2', 'glucose', 'heart_rate', 'sbp',
       'dbp', 'mbp', 'resp_rate', 'lactate', 'bun', 'creatinine',
       'bicarbonate'],
      dtype='object')


,care_unit,admission_type,admission_location,language,marital_status,race,hospital_expire_flag,gender,age,rr_interval,...,glucose,heart_rate,sbp,dbp,mbp,resp_rate,lactate,bun,creatinine,bicarbonate
0,neuro,ew_emer,physician_referral,english,single,white,0,1,72,434.0,...,169.400000,100.514286,130.513514,83.324324,98.081081,21.378378,1.933333,30.5,0.95,22.5
1,other,ew_emer,emergency_room,spanish,single,other,0,1,88,869.0,...,149.500000,95.333333,118.111111,47.703704,64.333333,16.771429,2.300000,20.0,1.00,24.0
2,cvicu,urgent,transfer_from_hospital,english,married,other,0,0,85,869.0,...,145.263158,79.420000,108.726190,50.107143,74.333333,16.879310,3.371429,22.0,1.10,19.5
3,cvicu,elective,physician_referral,english,married,white,0,0,79,833.0,...,158.315789,75.387097,113.878788,49.303030,67.484848,16.911765,1.466667,14.5,0.90,23.0
4,cvicu,ew_emer,emergency_room,english,single,other,1,0,37,869.0,...,140.444444,73.722222,151.235294,92.411765,106.529412,17.166667,3.900000,19.0,1.05,26.0
